# Stage 2 – Anomaly Detection + Full Pipeline Visualization

This notebook:
1. **Loads the Stage 1 Mask R-CNN** (teeth segmentation) from the uploaded dataset.
2. **Trains a Stage 2 anomaly-detection model** (Faster R-CNN) on the DENTEX-2023 dataset.
3. **Runs the full combined pipeline**: segment teeth with Stage 1, crop each tooth, run Stage 2 on it, render Stage 1 output with anomaly highlights.

**Dataset paths:**
- Stage 1 weights: `/kaggle/input/datasets/ethelrani/maskrcnn-teeth-stage1-weights/`
- Stage 2 training data: `/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/`

**Classes (Stage 2):**
- 0: background (implicit – no detection = Healthy)
- 1: Caries
- 2: Deep Caries
- 3: Periapical Lesion
- 4: Impacted Tooth


In [ ]:
# ================== CELL 1: SETUP & CONFIG ==================
print("[CELL 1] Starting setup...")

import os, json, random, copy, glob
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.ops import nms
import torchvision.transforms.functional as TF

print(f"  PyTorch:     {torch.__version__}")
print(f"  Torchvision: {torchvision.__version__}")
print(f"  CUDA:        {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU:         {torch.cuda.get_device_name(0)}")

# ── Fixed paths ────────────────────────────────────────────────────────────────
STAGE1_WEIGHTS_DIR   = "/kaggle/input/datasets/ethelrani/maskrcnn-teeth-stage1-weights"
STAGE1_BEST_WEIGHTS  = os.path.join(STAGE1_WEIGHTS_DIR, "maskrcnn_teeth_best.pth")
STAGE1_FINAL_WEIGHTS = os.path.join(STAGE1_WEIGHTS_DIR, "maskrcnn_teeth_final.pth")

DENTEX_DIR = "/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023"

print(f"\n[CELL 1] Checking Stage 1 weights directory...")
if os.path.isdir(STAGE1_WEIGHTS_DIR):
    files_in_s1 = os.listdir(STAGE1_WEIGHTS_DIR)
    print(f"  ✓ Stage 1 dir found. Contents: {files_in_s1}")
else:
    print(f"  ✗ Stage 1 dir NOT FOUND: {STAGE1_WEIGHTS_DIR}")
    print("  Scanning /kaggle/input/datasets/ethelrani/ ...")
    base = "/kaggle/input/datasets/ethelrani"
    if os.path.isdir(base):
        for entry in os.listdir(base):
            print(f"    {entry}")
    else:
        print(f"  ✗ /kaggle/input/datasets/ethelrani/ not found either!")

print(f"\n[CELL 1] Checking DENTEX-2023 root...")
if os.path.isdir(DENTEX_DIR):
    print(f"  ✓ DENTEX dir found")
    for root, dirs, files in os.walk(DENTEX_DIR):
        level = root.replace(DENTEX_DIR, '').count(os.sep)
        if level < 4:
            indent = '  ' * (level + 1)
            print(f"{indent}{os.path.basename(root)}/")
else:
    print(f"  ✗ DENTEX dir NOT FOUND: {DENTEX_DIR}")

# ── Auto-detect DENTEX subfolder (dashes vs underscores) ─────────────────────
def _find_subdir(base, *candidates):
    for c in candidates:
        p = os.path.join(base, c)
        if os.path.isdir(p):
            return p
    return None

_qed = _find_subdir(
    DENTEX_DIR,
    "quadrant_enumeration_disease",
    "quadrant-enumeration-disease",
    "training_data/training_data/quadrant_enumeration_disease",
    "training_data/training_data/quadrant-enumeration-disease",
)

if _qed is None:
    print("\n[CELL 1] ⚠  Could not auto-detect DENTEX subfolder. Full dir tree:")
    for root, dirs, _ in os.walk(DENTEX_DIR):
        for d in dirs:
            print(" ", os.path.join(root, d))
    raise RuntimeError("Set DENTEX_TRAIN_IMG / DENTEX_LABEL_DIR manually from the listing above.")

print(f"\n[CELL 1] ✓ quadrant_enumeration_disease dir: {_qed}")

DENTEX_TRAIN_IMG = (
    _find_subdir(_qed, "train/images", "xrays", "images", "train") or _qed
)
DENTEX_LABEL_DIR = (
    _find_subdir(_qed, "train/labels", "labels", "annotations", "train") or _qed
)
print(f"  DENTEX_TRAIN_IMG : {DENTEX_TRAIN_IMG}")
print(f"  DENTEX_LABEL_DIR : {DENTEX_LABEL_DIR}")

STAGE2_BEST  = "/kaggle/working/stage2_anomaly_best.pth"
STAGE2_FINAL = "/kaggle/working/stage2_anomaly_final.pth"

# ── Anomaly classes (DENTEX-2023 task 3) ─────────────────────────────────────
ANOMALY_CLASSES = {
    0: "background",
    1: "Caries",
    2: "Deep Caries",
    3: "Periapical Lesion",
    4: "Impacted Tooth",
}
NUM_ANOMALY_CLASSES = len(ANOMALY_CLASSES)  # 5

CONFIG2 = {
    "batch_size":     2,
    "lr":             1e-4,
    "epochs":         30,
    "conf_threshold": 0.5,
    "nms_iou":        0.3,
    "train_ratio":    0.75,
    "val_ratio":      0.15,
    "num_workers":    2,
    "crop_pad":       10,
    "min_crop_size":  32,
}

CONFIG1 = {
    "conf_threshold": 0.6,
    "nms_iou":        0.3,
    "padding":        20,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n[CELL 1] ✓ Device: {device}")

print("\n[CELL 1] Path existence check:")
for p, label in [
    (STAGE1_BEST_WEIGHTS,  "Stage 1 best weights (.pth)"),
    (STAGE1_FINAL_WEIGHTS, "Stage 1 final weights (.pth)"),
    (DENTEX_DIR,           "DENTEX-2023 root"),
    (DENTEX_TRAIN_IMG,     "DENTEX train images"),
    (DENTEX_LABEL_DIR,     "DENTEX train labels"),
]:
    status = '✓' if os.path.exists(p) else '✗ NOT FOUND'
    print(f"  {status}  {label}: {p}")

_n_png = len(glob.glob(os.path.join(DENTEX_TRAIN_IMG, "*.png")))
_n_jpg = len(glob.glob(os.path.join(DENTEX_TRAIN_IMG, "*.jpg")))
print(f"\n[CELL 1] Images found in DENTEX_TRAIN_IMG: {_n_png} PNG + {_n_jpg} JPG = {_n_png+_n_jpg} total")
print("[CELL 1] ✓ Setup complete.")


In [ ]:
# ================== CELL 2: DENTEX-2023 DATASET ==================
print("[CELL 2] Defining DentexAnomalyDataset...")

class DentexAnomalyDataset(Dataset):
    CATEGORY_MAP = {1: 1, 2: 2, 3: 3, 4: 4}

    def __init__(self, img_dir, label_dir, file_list=None):
        self.img_dir   = img_dir
        self.label_dir = label_dir
        all_imgs = sorted(
            glob.glob(os.path.join(img_dir, "*.png")) +
            glob.glob(os.path.join(img_dir, "*.jpg"))
        ) if file_list is None else [os.path.join(img_dir, f) for f in file_list]
        self.samples = []
        skipped_no_label = 0
        for img_path in all_imgs:
            base = os.path.splitext(os.path.basename(img_path))[0]
            lbl_txt  = os.path.join(label_dir, base + ".txt")
            lbl_json = os.path.join(label_dir, base + ".json")
            if os.path.exists(lbl_txt):
                self.samples.append((img_path, lbl_txt, "yolo"))
            elif os.path.exists(lbl_json):
                self.samples.append((img_path, lbl_json, "json"))
            else:
                skipped_no_label += 1
        print(f"  ✓ DentexAnomalyDataset: {len(self.samples)} labelled images  "
              f"(skipped {skipped_no_label} without labels)")
        if self.samples:
            print(f"  First sample: {os.path.basename(self.samples[0][0])}  format={self.samples[0][2]}")

    def __len__(self): return len(self.samples)

    def _load_yolo(self, path, w, h):
        boxes, labels = [], []
        for line in open(path):
            p = line.strip().split()
            if len(p) < 5: continue
            cid = int(p[0]) + 1
            if cid not in self.CATEGORY_MAP: continue
            cx, cy, bw, bh = map(float, p[1:5])
            x1, y1 = (cx - bw/2)*w, (cy - bh/2)*h
            x2, y2 = (cx + bw/2)*w, (cy + bh/2)*h
            if x2-x1<2 or y2-y1<2: continue
            boxes.append([x1,y1,x2,y2]); labels.append(self.CATEGORY_MAP[cid])
        return boxes, labels

    def _load_json(self, path, w, h):
        boxes, labels = [], []
        data = json.load(open(path))
        for ann in data.get("annotations", data.get("objects", [])):
            cid = ann.get("category_id", ann.get("label_id", -1))
            if cid not in self.CATEGORY_MAP: continue
            if "bbox" in ann:
                bx,by,bw,bh = ann["bbox"]; x1,y1,x2,y2 = bx,by,bx+bw,by+bh
            elif "points" in ann:
                pts=np.array(ann["points"]["exterior"])
                x1,y1,x2,y2=pts[:,0].min(),pts[:,1].min(),pts[:,0].max(),pts[:,1].max()
            else: continue
            if x2-x1<2 or y2-y1<2: continue
            boxes.append([x1,y1,x2,y2]); labels.append(self.CATEGORY_MAP[cid])
        return boxes, labels

    def __getitem__(self, idx):
        img_path, lbl_path, fmt = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        boxes, labels = self._load_yolo(lbl_path,w,h) if fmt=="yolo" else self._load_json(lbl_path,w,h)
        t = TF.to_tensor(img)
        if not boxes:
            target = {"boxes": torch.zeros((0,4),dtype=torch.float32),
                      "labels": torch.zeros((0,),dtype=torch.int64),
                      "image_id": torch.tensor([idx])}
        else:
            target = {"boxes": torch.tensor(boxes,dtype=torch.float32),
                      "labels": torch.tensor(labels,dtype=torch.int64),
                      "image_id": torch.tensor([idx])}
        return t, target

print("[CELL 2] ✓ DentexAnomalyDataset class defined.")

# Quick smoke-test with a single sample (if labels exist)
_smoke_ds = DentexAnomalyDataset(DENTEX_TRAIN_IMG, DENTEX_LABEL_DIR)
if len(_smoke_ds) > 0:
    _t, _tgt = _smoke_ds[0]
    print(f"[CELL 2] Smoke-test sample 0:")
    print(f"  image tensor shape : {_t.shape}")
    print(f"  boxes              : {_tgt['boxes'].shape}  -> {_tgt['boxes'].tolist()[:3]} ...")
    print(f"  labels             : {_tgt['labels'].tolist()[:10]} ...")
else:
    print("[CELL 2] ⚠  No labelled samples found. Check DENTEX_TRAIN_IMG / DENTEX_LABEL_DIR.")
print("[CELL 2] ✓ Done.")


In [ ]:
# ================== CELL 3: SPLIT & DATALOADERS ==================
print("[CELL 3] Building train/val/test splits...")

all_imgs = sorted([os.path.basename(p) for p in
    glob.glob(os.path.join(DENTEX_TRAIN_IMG,"*.png")) +
    glob.glob(os.path.join(DENTEX_TRAIN_IMG,"*.jpg"))])
random.seed(42); random.shuffle(all_imgs)
n_total=len(all_imgs)
n_train=int(n_total*CONFIG2["train_ratio"])
n_val  =int(n_total*CONFIG2["val_ratio"])
train_files=all_imgs[:n_train]
val_files  =all_imgs[n_train:n_train+n_val]
test_files =all_imgs[n_train+n_val:]
print(f"  Total images : {n_total}")
print(f"  Train split  : {len(train_files)}")
print(f"  Val split    : {len(val_files)}")
print(f"  Test split   : {len(test_files)}")

train_ds = DentexAnomalyDataset(DENTEX_TRAIN_IMG, DENTEX_LABEL_DIR, file_list=train_files)
val_ds   = DentexAnomalyDataset(DENTEX_TRAIN_IMG, DENTEX_LABEL_DIR, file_list=val_files)
test_ds  = DentexAnomalyDataset(DENTEX_TRAIN_IMG, DENTEX_LABEL_DIR, file_list=test_files)
print(f"  Labelled train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}")

def collate_fn(batch): return tuple(zip(*batch))

train_loader2 = DataLoader(train_ds, batch_size=CONFIG2["batch_size"], shuffle=True,
                           collate_fn=collate_fn, num_workers=CONFIG2["num_workers"])
val_loader2   = DataLoader(val_ds,   batch_size=CONFIG2["batch_size"], shuffle=False,
                           collate_fn=collate_fn, num_workers=CONFIG2["num_workers"])
test_loader2  = DataLoader(test_ds,  batch_size=1, shuffle=False,
                           collate_fn=collate_fn, num_workers=CONFIG2["num_workers"])
print(f"  Train batches: {len(train_loader2)}  Val batches: {len(val_loader2)}  Test batches: {len(test_loader2)}")

# Verify first batch loads cleanly
print("[CELL 3] Verifying first train batch...")
_imgs_b, _tgts_b = next(iter(train_loader2))
print(f"  Batch size returned : {len(_imgs_b)}")
print(f"  Image 0 tensor shape: {_imgs_b[0].shape}")
print(f"  Target 0 boxes      : {_tgts_b[0]['boxes'].shape}")
print(f"  Target 0 labels     : {_tgts_b[0]['labels'].tolist()}")
print("[CELL 3] ✓ Dataloaders ready.")


In [ ]:
# ================== CELL 4: STAGE 2 MODEL (FASTER R-CNN) ==================
print("[CELL 4] Building Stage 2 Faster R-CNN...")

def get_anomaly_detector(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)
    return model

stage2_model = get_anomaly_detector(NUM_ANOMALY_CLASSES).to(device)
optimizer2   = torch.optim.Adam(stage2_model.parameters(), lr=CONFIG2["lr"])
scheduler2   = torch.optim.lr_scheduler.StepLR(optimizer2, step_size=10, gamma=0.5)

# Parameter count
_total_params = sum(p.numel() for p in stage2_model.parameters())
_train_params = sum(p.numel() for p in stage2_model.parameters() if p.requires_grad)
print(f"  ✓ Stage 2 Faster R-CNN built on {device}")
print(f"  Total params    : {_total_params:,}")
print(f"  Trainable params: {_train_params:,}")
print(f"  Classes ({NUM_ANOMALY_CLASSES}): {list(ANOMALY_CLASSES.values())}")
print(f"  Optimizer       : Adam  lr={CONFIG2['lr']}")
print(f"  LR scheduler    : StepLR  step=10  gamma=0.5")
print("[CELL 4] ✓ Model ready.")


In [ ]:
# ================== CELL 5: TRAINING STAGE 2 ==================
print(f"[CELL 5] Training Stage 2 for {CONFIG2['epochs']} epochs...")
print(f"  Train batches: {len(train_loader2)}  Val batches: {len(val_loader2)}")
print(f"  Batch size   : {CONFIG2['batch_size']}  LR: {CONFIG2['lr']}")
print("  "+"-"*55)

best_val2=float("inf"); train_losses2=[]; val_losses2=[]

for epoch in range(CONFIG2["epochs"]):
    # ── TRAIN ──────────────────────────────────────────────────────────────
    stage2_model.train(); ep_tr=0.0; n_tr=0; skipped_tr=0
    for batch_i, (imgs, tgts) in enumerate(train_loader2):
        valid=[(i,t) for i,t in zip(imgs,tgts) if t["boxes"].shape[0]>0]
        if not valid:
            skipped_tr+=1; continue
        im,tg=zip(*valid)
        im=[x.to(device) for x in im]
        tg=[{k:v.to(device) for k,v in t.items()} for t in tg]
        loss_dict=stage2_model(im,tg)
        loss=sum(loss_dict.values())
        optimizer2.zero_grad(); loss.backward(); optimizer2.step()
        ep_tr+=loss.item(); n_tr+=1
    avg_tr=ep_tr/max(1,n_tr); train_losses2.append(avg_tr)

    # ── VALIDATION ─────────────────────────────────────────────────────────
    stage2_model.train(); ep_vl=0.0; n_vl=0; skipped_vl=0
    with torch.no_grad():
        for imgs,tgts in val_loader2:
            valid=[(i,t) for i,t in zip(imgs,tgts) if t["boxes"].shape[0]>0]
            if not valid:
                skipped_vl+=1; continue
            im,tg=zip(*valid)
            im=[x.to(device) for x in im]
            tg=[{k:v.to(device) for k,v in t.items()} for t in tg]
            ep_vl+=sum(stage2_model(im,tg).values()).item(); n_vl+=1
    avg_vl=ep_vl/max(1,n_vl); val_losses2.append(avg_vl)
    scheduler2.step()

    saved_flag = ""
    if avg_vl<best_val2:
        best_val2=avg_vl
        torch.save({"epoch":epoch+1,"model_state_dict":stage2_model.state_dict(),
                    "optimizer_state_dict":optimizer2.state_dict(),"val_loss":best_val2},STAGE2_BEST)
        saved_flag = "  ← best saved"

    print(f"  Epoch {epoch+1:03d}/{CONFIG2['epochs']}  "
          f"Train: {avg_tr:.4f} ({n_tr} batches, {skipped_tr} empty skipped)  "
          f"Val: {avg_vl:.4f} ({n_vl} batches){saved_flag}")

torch.save(stage2_model.state_dict(), STAGE2_FINAL)
print(f"\n[CELL 5] ✓ Training complete! Best val_loss={best_val2:.4f}")
print(f"  Best weights  → {STAGE2_BEST}")
print(f"  Final weights → {STAGE2_FINAL}")

plt.figure(figsize=(10,4))
plt.plot(train_losses2,label="Train",color="#e74c3c")
plt.plot(val_losses2,  label="Val",  color="#3498db")
plt.title("Stage 2 – Training Curves"); plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.tight_layout()
plt.savefig("/kaggle/working/stage2_training_curve.png",dpi=150)
print("[CELL 5] ✓ Training curve saved → /kaggle/working/stage2_training_curve.png")
plt.show()


In [ ]:
# ================== CELL 6: RELOAD BEST STAGE 2 WEIGHTS ==================
print("[CELL 6] Reloading best Stage 2 weights...")

if not os.path.exists(STAGE2_BEST):
    raise FileNotFoundError(f"[CELL 6] ✗ Best checkpoint not found: {STAGE2_BEST}  "
                            "Did Cell 5 training complete successfully?")

ckpt2 = torch.load(STAGE2_BEST, map_location=device)
stage2_model.load_state_dict(ckpt2["model_state_dict"])
stage2_model.eval()
print(f"  ✓ Loaded epoch     : {ckpt2['epoch']}")
print(f"  ✓ Best val_loss    : {ckpt2['val_loss']:.4f}")
print(f"  ✓ Checkpoint path  : {STAGE2_BEST}")
print("[CELL 6] ✓ Stage 2 model in eval mode.")


In [ ]:
# ================== CELL 7: STAGE 1 HELPERS ==================
print("[CELL 7] Loading Stage 1 Mask R-CNN...")
print(f"  Looking for weights in: {STAGE1_WEIGHTS_DIR}")

if os.path.isdir(STAGE1_WEIGHTS_DIR):
    print(f"  Contents: {os.listdir(STAGE1_WEIGHTS_DIR)}")
else:
    print(f"  ✗ Directory not found: {STAGE1_WEIGHTS_DIR}")

def get_stage1_model(num_classes=33):
    m = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")
    in_f = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(in_f, num_classes)
    in_fm = m.roi_heads.mask_predictor.conv5_mask.in_channels
    m.roi_heads.mask_predictor = MaskRCNNPredictor(in_fm, 256, num_classes)
    return m

stage1_model = get_stage1_model().to(device)
print(f"  Stage 1 Mask R-CNN created (num_classes=33) on {device}")

if os.path.exists(STAGE1_BEST_WEIGHTS):
    ck1 = torch.load(STAGE1_BEST_WEIGHTS, map_location=device)
    state = ck1.get("model_state_dict", ck1)
    stage1_model.load_state_dict(state)
    print(f"  ✓ Stage 1 BEST weights loaded")
    print(f"    epoch    : {ck1.get('epoch', 'N/A')}")
    print(f"    val_loss : {ck1.get('val_loss', 'N/A')}")
elif os.path.exists(STAGE1_FINAL_WEIGHTS):
    stage1_model.load_state_dict(torch.load(STAGE1_FINAL_WEIGHTS, map_location=device))
    print("  ✓ Stage 1 FINAL weights loaded (best not found)")
else:
    print(f"  ✗ No Stage 1 weights found!")
    print(f"    Checked: {STAGE1_BEST_WEIGHTS}")
    print(f"    Checked: {STAGE1_FINAL_WEIGHTS}")
    raise FileNotFoundError("Stage 1 weights missing. Check STAGE1_WEIGHTS_DIR path.")

stage1_model.eval()
print("  ✓ Stage 1 model in eval mode")

# ── Stage 1 inference helpers ──────────────────────────────────────────────
def _nms_pred(pred, iou=CONFIG1["nms_iou"]):
    if len(pred["boxes"])==0: return pred
    keep=nms(pred["boxes"],pred["scores"],iou)
    return {k:v[keep] for k,v in pred.items()}

def run_stage1(img_tensor):
    with torch.no_grad():
        raw=stage1_model(img_tensor.to(device).unsqueeze(0))[0]
    keep=raw["scores"]>=CONFIG1["conf_threshold"]
    filt=_nms_pred({k:v[keep] for k,v in raw.items()})
    return {k:filt[k].cpu().numpy() for k in ["boxes","labels","masks","scores"]}

def crop_to_teeth(img_np, preds, pad=CONFIG1["padding"]):
    b=preds["boxes"]
    if len(b)==0: return img_np,None
    x1,y1=max(0,int(b[:,0].min())-pad),max(0,int(b[:,1].min())-pad)
    x2,y2=min(img_np.shape[1],int(b[:,2].max())+pad),min(img_np.shape[0],int(b[:,3].max())+pad)
    return img_np[y1:y2,x1:x2],(x1,y1,x2,y2)

def color_teeth_fn(image, preds, crop_box=None):
    colored=cv2.cvtColor(image,cv2.COLOR_GRAY2RGB) if image.ndim==2 else image.copy()
    palette=[[255,60,60],[60,255,60],[60,60,255],[255,220,0],[220,0,255],[0,220,255],
             [255,140,0],[140,0,255],[255,0,140],[0,255,140],[140,255,0],[0,140,255]]
    for i,(mask,box) in enumerate(zip(preds["masks"],preds["boxes"])):
        mb=(mask[0]>0.5).astype(np.uint8)
        if crop_box:
            x1,y1,x2,y2=crop_box; mb=mb[y1:y2,x1:x2]
        if mb.shape!=colored.shape[:2]: continue
        color=palette[i%len(palette)]
        ov=colored.copy()
        for c in range(3): ov[:,:,c]=np.where(mb==1,color[c],ov[:,:,c])
        colored=cv2.addWeighted(colored,0.3,ov,0.7,0)
        cnts,_=cv2.findContours(mb,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(colored,cnts,-1,color,2)
    return colored

def compute_center(preds, shape):
    if len(preds["boxes"])==0: return shape[1]//2,shape[0]//2
    cy=float(np.median([(b[1]+b[3])/2 for b in preds["boxes"]]))
    cx=float(np.median([(b[0]+b[2])/2 for b in preds["boxes"]]))
    return int(cx),int(cy)

def assign_quadrants(preds, center):
    cx,cy=center
    quads={"Q1":[],"Q2":[],"Q3":[],"Q4":[]}
    for i,box in enumerate(preds["boxes"]):
        tx,ty=(box[0]+box[2])/2,(box[1]+box[3])/2
        q=("Q1" if tx<cx else "Q2") if ty<cy else ("Q3" if tx<cx else "Q4")
        quads[q].append({"idx":i,"centroid":(int(tx),int(ty)),"box":box,
                          "label":preds["labels"][i],"score":preds["scores"][i],
                          "mask":preds["masks"][i]})
    for q,teeth in quads.items():
        quads[q]=sorted(teeth,key=lambda t:t["centroid"][0],reverse=(q in ("Q1","Q3")))
    return quads

def number_teeth(quads):
    starts={"Q1":11,"Q2":21,"Q3":31,"Q4":41}
    for q,teeth in quads.items():
        for rank,tooth in enumerate(teeth):
            tooth["number"]=starts[q]+rank; tooth["quadrant"]=q
    return quads

print("[CELL 7] ✓ All Stage 1 helper functions defined.")


In [ ]:
# ================== CELL 8: PER-TOOTH ANOMALY DETECTION ==================
print("[CELL 8] Defining per-tooth anomaly detection helpers...")

def crop_single_tooth(img_np, box, pad=CONFIG2["crop_pad"]):
    x1,y1,x2,y2=box; H,W=img_np.shape[:2]
    crop = img_np[
        max(0,int(y1)-pad):min(H,int(y2)+pad),
        max(0,int(x1)-pad):min(W,int(x2)+pad)
    ]
    return crop, None

def run_stage2_on_crop(crop_np, tooth_num=None):
    if crop_np.shape[0]<CONFIG2["min_crop_size"] or crop_np.shape[1]<CONFIG2["min_crop_size"]:
        if tooth_num: print(f"    Tooth {tooth_num}: crop too small {crop_np.shape[:2]}, skipping")
        return []
    rgb = cv2.cvtColor(crop_np,cv2.COLOR_GRAY2RGB) if crop_np.ndim==2 else crop_np.copy()
    t = TF.to_tensor(Image.fromarray(rgb)).to(device)
    stage2_model.eval()
    with torch.no_grad():
        pred = stage2_model([t])[0]
    results = []
    for box,lbl,score in zip(pred["boxes"].cpu().numpy(),
                              pred["labels"].cpu().numpy(),
                              pred["scores"].cpu().numpy()):
        if score < CONFIG2["conf_threshold"]: continue
        results.append({
            "label":      int(lbl),
            "label_name": ANOMALY_CLASSES.get(int(lbl),"Unknown"),
            "score":      float(score),
            "box":        box.tolist()
        })
    return results

def detect_anomalies(img_np, numbered_quads):
    anomaly_map = {}
    total_teeth = sum(len(v) for v in numbered_quads.values())
    print(f"  Running Stage 2 on {total_teeth} individual tooth crops...")
    checked = 0
    for q_name, teeth in numbered_quads.items():
        for tooth in teeth:
            crop, _ = crop_single_tooth(img_np, tooth["box"])
            dets = run_stage2_on_crop(crop, tooth_num=tooth.get("number"))
            checked += 1
            if dets:
                anomaly_map[tooth["number"]] = {
                    "quadrant": q_name,
                    "tooth":    tooth,
                    "anomalies":dets
                }
                for d in dets:
                    print(f"    ⚠  Tooth {tooth['number']:>2} ({q_name}): "
                          f"{d['label_name']}  conf={d['score']:.2f}")
    print(f"  Checked {checked} teeth  |  Anomalous: {len(anomaly_map)}  |  "
          f"Healthy: {checked - len(anomaly_map)}")
    return anomaly_map

print("[CELL 8] ✓ Per-tooth anomaly detection helpers defined.")


In [ ]:
# ================== CELL 9: FULL PIPELINE + VISUALIZATION ==================
print("[CELL 9] Defining full pipeline visualization function...")

HIGHLIGHT_COLOR   = (255,  50,  50)  # red   – anomalous
QUAD_LINE_COLOR   = (255, 230,   0)  # yellow quadrant lines
NUM_NORMAL_COLOR  = (  0, 255, 255)  # cyan  – healthy tooth numbers
NUM_ANOMALY_COLOR = (255,  50,  50)  # red   – anomalous tooth numbers

def run_full_pipeline(image_source, save_path=None):
    print(f"\n  [pipeline] Loading image...")
    if isinstance(image_source, str):
        img_np = np.array(Image.open(image_source).convert("RGB"))
        print(f"    source  : {os.path.basename(image_source)}")
    elif isinstance(image_source, np.ndarray):
        img_np = cv2.cvtColor(image_source,cv2.COLOR_GRAY2RGB) if image_source.ndim==2 else image_source.copy()
    else:
        img_np = np.array(image_source.convert("RGB"))
    print(f"    img shape: {img_np.shape}  dtype: {img_np.dtype}")

    img_tensor = TF.to_tensor(Image.fromarray(img_np))

    # ── Stage 1 ────────────────────────────────────────────────────────────
    print(f"  [pipeline] Running Stage 1 (Mask R-CNN teeth segmentation)...")
    preds_s1 = run_stage1(img_tensor)
    n_teeth = len(preds_s1['labels'])
    print(f"    ✓ {n_teeth} teeth detected  (conf≥{CONFIG1['conf_threshold']})")
    if n_teeth > 0:
        print(f"    Score range: {preds_s1['scores'].min():.2f} – {preds_s1['scores'].max():.2f}")
        print(f"    Box range  : x1={preds_s1['boxes'][:,0].min():.0f}  "
              f"x2={preds_s1['boxes'][:,2].max():.0f}  "
              f"y1={preds_s1['boxes'][:,1].min():.0f}  "
              f"y2={preds_s1['boxes'][:,3].max():.0f}")

    center         = compute_center(preds_s1, img_np.shape)
    quads          = assign_quadrants(preds_s1, center)
    numbered_quads = number_teeth(quads)
    print(f"    Center: {center}")
    for q, teeth in numbered_quads.items():
        nums = [t['number'] for t in teeth]
        print(f"    {q}: {len(teeth)} teeth  numbers={nums}")

    cropped_np, crop_box = crop_to_teeth(img_np, preds_s1)
    print(f"    Crop box: {crop_box}  → cropped shape: {cropped_np.shape}")
    colored_np = color_teeth_fn(cropped_np.copy(), preds_s1, crop_box)
    print(f"    Colored image shape: {colored_np.shape}")

    # ── Stage 2 ────────────────────────────────────────────────────────────
    print(f"  [pipeline] Running Stage 2 (Faster R-CNN anomaly detection)...")
    anomaly_map = detect_anomalies(img_np, numbered_quads)

    # ── Build annotated image ───────────────────────────────────────────────
    print(f"  [pipeline] Building annotated result image...")
    result_img = colored_np.copy(); h,w = result_img.shape[:2]
    ox = crop_box[0] if crop_box else 0
    oy = crop_box[1] if crop_box else 0
    cx_c = center[0]-ox; cy_c = center[1]-oy

    cv2.line(result_img,(cx_c,0),(cx_c,h),QUAD_LINE_COLOR,4)
    cv2.line(result_img,(0,cy_c),(w,cy_c),QUAD_LINE_COLOR,4)

    for ql,pos in {"Q1":(10,40),"Q2":(w-80,40),"Q3":(10,h-20),"Q4":(w-80,h-20)}.items():
        cv2.putText(result_img,ql,pos,cv2.FONT_HERSHEY_SIMPLEX,1.4,(0,0,0),5)
        cv2.putText(result_img,ql,pos,cv2.FONT_HERSHEY_SIMPLEX,1.4,(255,255,255),2)

    for q_name,teeth in numbered_quads.items():
        for tooth in teeth:
            tx=tooth["centroid"][0]-ox; ty=tooth["centroid"][1]-oy
            if not(0<=tx<w and 0<=ty<h): continue
            num=tooth["number"]; is_a=num in anomaly_map
            if is_a:
                bx1=max(0,int(tooth["box"][0])-ox-5); by1=max(0,int(tooth["box"][1])-oy-5)
                bx2=min(w,int(tooth["box"][2])-ox+5); by2=min(h,int(tooth["box"][3])-oy+5)
                cv2.rectangle(result_img,(bx1,by1),(bx2,by2),HIGHLIGHT_COLOR,6)
                cv2.rectangle(result_img,(bx1+4,by1+4),(bx2-4,by2-4),(255,150,0),2)
            nc = NUM_ANOMALY_COLOR if is_a else NUM_NORMAL_COLOR
            cv2.putText(result_img,str(num),(tx-20,ty+15),cv2.FONT_HERSHEY_SIMPLEX,1.2,(0,0,0),5)
            cv2.putText(result_img,str(num),(tx-20,ty+15),cv2.FONT_HERSHEY_SIMPLEX,1.2,nc,2)

    # ── Full tooth report (healthy + anomalous) ─────────────────────────────
    all_teeth_flat = sorted(
        [t for teeth in numbered_quads.values() for t in teeth],
        key=lambda t: t["number"]
    )
    lines = ["FULL TOOTH REPORT", "─" * 34]
    n_healthy = 0; n_anomalous = 0
    for tooth in all_teeth_flat:
        tnum = tooth["number"]
        if tnum in anomaly_map:
            for a in anomaly_map[tnum]["anomalies"]:
                lines.append(f"  ⚠ Tooth {tnum:>2}  ({tooth['quadrant']}):  "
                             f"{a['label_name']}  [{a['score']*100:.0f}%]")
            n_anomalous += 1
        else:
            lines.append(f"  ✓ Tooth {tnum:>2}  ({tooth['quadrant']}):  Healthy")
            n_healthy += 1
    lines.append("─" * 34)
    lines.append(f"  Total: {len(all_teeth_flat)} teeth  |  "
                 f"Healthy: {n_healthy}  |  Anomalous: {n_anomalous}")

    print("  [pipeline] " + "\n  [pipeline] ".join(lines))

    # ── 4-panel Figure ────────────────────────────────────────────────────────
    print(f"  [pipeline] Rendering 4-panel figure...")
    fig=plt.figure(figsize=(22,18),facecolor="#1a1a2e")
    gs=fig.add_gridspec(2,2,hspace=0.08,wspace=0.08,left=0.02,right=0.98,top=0.93,bottom=0.02)
    tkw=dict(fontsize=14,fontweight="bold",color="white",fontfamily="monospace",pad=8)

    ax1=fig.add_subplot(gs[0,0]); ax1.imshow(img_np,cmap="gray");    ax1.set_title("Original Panoramic X-ray",**tkw);                                   ax1.axis("off")
    ax2=fig.add_subplot(gs[0,1]); ax2.imshow(cropped_np,cmap="gray"); ax2.set_title("Cropped to Teeth Region  (Stage 1)",**tkw);                          ax2.axis("off")
    ax3=fig.add_subplot(gs[1,0]); ax3.imshow(colored_np);             ax3.set_title("Coloured Teeth Segmentation  (Stage 1)",**tkw);                      ax3.axis("off")
    ax4=fig.add_subplot(gs[1,1]); ax4.imshow(result_img);             ax4.set_title("Quadrants + Tooth Numbers + Anomaly Highlights  (Stage 1+2)",**tkw); ax4.axis("off")

    fig.suptitle("Dental X-ray Analysis Pipeline",fontsize=20,fontweight="bold",
                 color="white",fontfamily="monospace",y=0.97)
    fig.text(0.02,-0.01,"\n".join(lines),fontsize=9,color="#f0f0f0",fontfamily="monospace",va="top",
             bbox=dict(boxstyle="round,pad=0.5",facecolor="#0d0d1a",edgecolor="#ff3232",linewidth=1.5))

    np_norm=np.array(NUM_NORMAL_COLOR)/255.; np_anom=np.array(HIGHLIGHT_COLOR)/255.
    fig.legend(handles=[mpatches.Patch(color=np_norm,label="Healthy tooth"),
                        mpatches.Patch(color=np_anom,label="Anomalous tooth")],
               loc="lower right",fontsize=11,facecolor="#1a1a2e",edgecolor="gray",
               labelcolor="white",framealpha=0.9)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path,dpi=150,bbox_inches="tight",facecolor=fig.get_facecolor())
        print(f"  [pipeline] ✓ Saved → {save_path}")
    plt.show()
    return fig, anomaly_map

print("[CELL 9] ✓ run_full_pipeline() defined.")


In [ ]:
# ================== CELL 10: RUN ON TEST SAMPLES ==================
print("[CELL 10] Running pipeline on up to 3 test images from DENTEX...")

test_imgs = sorted(
    glob.glob(os.path.join(DENTEX_TRAIN_IMG,"*.png"))[:5] +
    glob.glob(os.path.join(DENTEX_TRAIN_IMG,"*.jpg"))[:5]
)[:3]

print(f"  Found {len(test_imgs)} test images:")
for p in test_imgs:
    print(f"    {os.path.basename(p)}")

for i, img_path in enumerate(test_imgs):
    print(f"\n{'='*60}")
    print(f"  Image {i+1}/{len(test_imgs)}: {os.path.basename(img_path)}")
    print(f"{'='*60}")
    save_p = f"/kaggle/working/dental_analysis_{i+1}.png"
    fig, amap = run_full_pipeline(img_path, save_path=save_p)
    plt.close(fig)
    print(f"  Output: {save_p}")

print("\n[CELL 10] ✓ All test outputs saved to /kaggle/working/")
print("  Files:", os.listdir("/kaggle/working/"))


In [ ]:
# ================== CELL 11: RUN ON CUSTOM / VALIDATION IMAGE ==================
print("[CELL 11] Running pipeline on a DENTEX validation xray (no labels needed)...")

IMAGE_PATH = "/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/validation_data/validation_data/quadrant_enumeration_disease/xrays"

if os.path.isdir(IMAGE_PATH):
    _candidates = glob.glob(os.path.join(IMAGE_PATH,"*.png")) + glob.glob(os.path.join(IMAGE_PATH,"*.jpg"))
    _candidates = sorted(_candidates)
    print(f"  Found {len(_candidates)} images in validation xrays dir.")
    if _candidates:
        IMAGE_PATH = _candidates[0]
        print(f"  Auto-selected: {os.path.basename(IMAGE_PATH)}")
    else:
        print("  ⚠  No images found in validation dir. Change IMAGE_PATH manually.")
else:
    print(f"  Using provided path: {IMAGE_PATH}")

if os.path.exists(IMAGE_PATH) and os.path.isfile(IMAGE_PATH):
    print(f"  ✓ Image found: {IMAGE_PATH}")
    fig, anomaly_map = run_full_pipeline(IMAGE_PATH, save_path="/kaggle/working/dental_analysis_custom.png")
    plt.close(fig)
    print(f"\n[CELL 11] ✓ Custom analysis saved → /kaggle/working/dental_analysis_custom.png")
else:
    print(f"  ✗ Image not found or is a directory: {IMAGE_PATH}")
    print("  Change IMAGE_PATH to the full path of a panoramic X-ray .png/.jpg file.")

print("\n[CELL 11] Final /kaggle/working/ contents:")
for f in sorted(os.listdir("/kaggle/working/")):
    size_kb = os.path.getsize(os.path.join("/kaggle/working/",f)) // 1024
    print(f"  {f}  ({size_kb} KB)")
